In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.model_selection import GroupShuffleSplit

In [3]:
source = pd.read_json("python_dataset.jsonl", lines=True)

In [4]:
df = source

df = df.head(1000)

df = df.drop(columns="docstring")

# Convert df from wide format to long format. Originally it was idx, human_code, gpt_code, qwen_code ...
# But i need seperate rows per code snippet. With melt I transform it to idx, code, source
df = pd.melt(
    df,
    id_vars="hm_index",
    value_vars=["human_code", "chatgpt_code", "dsc_code", "qwen_code"],
    var_name="source",
    value_name="code",
)

df = df.dropna(axis=0)

# Add new column thats 0 or 1 depending on source. Human-written is 0, AI-generated is 1
df["ai_generated"] = df["source"].ne("human_code").astype(int)

df = df.rename(columns={"hm_index": "task_id"})

# Simplify task_id values and sort df
df["task_id"] = df.groupby("task_id").ngroup() + 1
df = df.sort_values(by=["task_id"])

# Move colum order so source comes last
df = df.iloc[:, [0, 2, 3, 1]]


In [ ]:
# Split code into input X and ouput y
X = df["code"]
y = df["ai_generated"]

# The df has multiple rows per coding task. e.g. task 1 has 4 rows (human-, gpt-, qwen- and dsc- code snippets)
# with GroupShuffleShift I split the the rows but dont put rows with the same task_id into train AND test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in gss.split(df, groups=df["task_id"]):
    train_X, train_y = X.iloc[train_idx], y.iloc[train_idx]
    test_X, test_y = X.iloc[test_idx], y.iloc[test_idx]

# schneidung = train_X["task_id"].isin(test_X["task_id"]).any()
# print(schneidung)

# CountVectorizer:
# Takes documents and makes vectors with the count of the tokens as dimensions. Simple example: "hello cat. Bye cat." would become (1, 2, 1)
# Step 1 fit: finds all unique tokens and builds a vocabulary
# Step 2 transform: counts how many times the tokens appear and creates a vector per document

# TfidfTransformer:
# Takes count matrix (multiple vectors) and compares the frequency of each token with the frequency of that token in all the vectors.
# TF: How many times does this token appear in a document?
# IDF: In how many different documents does this token appear in?
# Step 1 fit: calculates the inverse document frequency / weights. So (num of documents / num of document where token appears)
# Step 2 transform: Looks at a single document and compares the tokens in the count vector with
# the corresponding idf weight calculated in Step 1. Forms and normalises a vector.


# Init a count vectorizer. make a count vector for each row in the training data (only code)
cv = CountVectorizer()
train_count = cv.fit_transform(train_X)

# Init a tfidf transformer. pass all the count vectors into transformer and get a vector for each row in the training data
transformer = TfidfTransformer(smooth_idf=True, use_idf=True)
train_tdidf = transformer.fit_transform(train_count)


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 82280 stored elements and shape (3200, 15592)>
  Coords	Values
  (0, 158)	0.03898821914040342
  (0, 299)	0.09918937565697093
  (0, 2286)	0.08834125778879515
  (0, 2303)	0.09918937565697093
  (0, 2577)	0.08277680690965171
  (0, 3210)	0.06156386953348764
  (0, 3977)	0.021354224775757984
  (0, 4007)	0.3533650311551806
  (0, 4168)	0.01213122831324264
  (0, 5156)	0.09918937565697093
  (0, 5186)	0.05660019947739899
  (0, 5661)	0.09918937565697093
  (0, 5724)	0.03445765003444415
  (0, 5847)	0.051715663099517466
  (0, 5852)	0.09098309128331132
  (0, 6300)	0.04549154564165566
  (0, 6803)	0.26502377336638544
  (0, 7119)	0.06372240466781634
  (0, 7207)	0.14158351054218907
  (0, 7209)	0.14158351054218907
  (0, 7217)	0.019022090706548745
  (0, 7248)	0.09558360700172452
  (0, 7262)	0.30919440226078304
  (0, 7263)	0.09918937565697093
  (0, 7265)	0.049594687828485466
  :	:
  (3198, 5357)	0.2782340988909318
  (3198, 5672)	0.05489622585577117